In [1]:
# Installing the 'Toolbox' (Libraries)
!pip install -q --upgrade langchain langchain-community langchain-core chromadb pypdf langchain-google-genai unstructured python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 13.4 MB/s eta 0:00:

# 2. Load the PDF and Split it into Chunks

In [3]:
from langchain_community.document_loaders import PyPDFLoader, UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF File paths
pdf_paths = [
    "/content/smart_hr_document_assistant/Sickness-And-Absence-Policy.pdf",
    "/content/smart_hr_document_assistant/Fixed-Term-Employment-Contract.pdf"
]

# DOCX File paths
doc_paths = [
    "/content/smart_hr_document_assistant/Harassment-and-bullying-policy-3.docx",
    "/content/smart_hr_document_assistant/Employee-Expenses-Policy.docx"
]

documents = []

# Load PDFs
for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = path.split("/")[-1]
        doc.metadata["type"] = "pdf"

    documents.extend(docs)

# Load DOCX files
for path in doc_paths:
    loader = UnstructuredWordDocumentLoader(path)
    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = path.split("/")[-1]
        doc.metadata["type"] = "docx"
        doc.metadata["page"] = doc.metadata.get("page", "N/A")  # normalize

    documents.extend(docs)

print(f"Loaded {len(documents)} total documents/pages")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from all documents")

Loaded 15 total documents/pages
Created 69 chunks from all documents


# 3. Create the Embeddings (Turning text into numbers using Gemini)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

# 4. Create/Update the Vector DB (using the chunks from your previous step)

In [5]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)

# 5. Creating the langchain pipeline and calling the Gemini 2.5 Flash model to give response along with citations

In [7]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Using gemini 2.5 flash model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

# Create retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below. If the answer is not in the context, say "Not found in provided documents".

Context:
{context}

Question:
{question}
""")

# Function to format docs into context string
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# New RAG chain (keeps docs!)
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
        "docs": retriever
    }
    | RunnableLambda(lambda x: {
        "answer": llm.invoke(prompt.format(**x)).content,
        "docs": x["docs"]
    })
)

# Query
query = "What all expenses in UK can be reimbursed?"
result = rag_chain.invoke(query)

# Print answer
print("---- ANSWER ----")
print(result["answer"])

# Extract and print sources
print("\n---- SOURCES ----")
seen = set()

for doc in result["docs"]:
    source = doc.metadata.get("source", "Unknown")
    page = doc.metadata.get("page", "N/A")

    key = (source, page)
    if key not in seen:
        print(f"{source} (Page {page})")
        seen.add(key)

---- ANSWER ----
Based on the context provided, the following expenses can be reimbursed in the UK:

*   **Subsistence:** Evening meal, Breakfast, and Lunch.
*   **Overnight incidental expenses:** (e.g., drinks, newspapers, laundry, and personal telephone calls) for overnight stays within the UK.
*   **Entertainment and hospitality expenses.**
*   **Travel:** Specifically, Rail Travel Expenses for journeys within the UK.
*   **Accommodation.**
*   **Training.**
*   **Membership of professional bodies.**
*   **Business-related personal expenditure:** Including reasonable telephone calls made for business purposes using a personal telephone, and reasonable personal expenditure on equipment for business purposes.

---- SOURCES ----
Employee-Expenses-Policy.docx (Page N/A)
